In [1]:
"""
================================================================================
Data Poisoning / Backdoor — FedAvg vs. U-Split (NO CAWA)
================================================================================
Comparing standard FedAvg against U-Shaped Split Learning under a
data poisoning and backdoor attack without defensive aggregation.
Output: vqarad_poisoning_comparative_results.xlsx
================================================================================
"""
import re
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets", "openpyxl", "tqdm"])
import os, random, time, copy; import numpy as np; from collections import Counter; from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')
import torch, torch.nn as nn, torch.nn.functional as F; from torch.utils.data import Dataset, DataLoader

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(f"Device: {device}")
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
OUTPUT_DIR="/kaggle/working/"; os.makedirs(OUTPUT_DIR,exist_ok=True)

D=256; VOCAB_SIZE=30522; MAX_SEQ=64; HEADS=4; DROP=0.15; CBAM_BLOCKS=3; TEXT_ENC_LAYERS=2; TEXT_REFINE_LAYERS=2; FUSE_LAYERS=4
NUM_CLIENTS=5; ROUNDS=20; LOCAL_EP=3; BS=32; FED_LR=3e-4; WD=1e-4
MALICIOUS_CLIENT=0; LABEL_FLIP_RATE=0.30; BACKDOOR_RATE=0.20; TRIGGER_SIZE=8; BACKDOOR_TARGET=1


def normalize_answer_slake(ans: str) -> str:
    ans = ans.strip().lower()
    ans = re.sub(r'[^\w\s\-/.,]', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    yes_set = {'yes', 'yes.', 'yeah', 'yep', 'y', 'correct', 'true'}
    no_set  = {'no', 'no.', 'nope', 'n', 'false', 'incorrect', 'negative',
               'none', 'not sure'}
    if ans in yes_set:
        return 'yes'
    if ans in no_set:
        return 'no'

    word_to_num = {'zero': '0', 'one': '1', 'two': '2', 'three': '3',
                   'four': '4', 'five': '5', 'six': '6', 'seven': '7',
                   'eight': '8', 'nine': '9', 'ten': '10'}
    if ans in word_to_num:
        return word_to_num[ans]

    modality_map = {
        'ct scan': 'ct', 'ct': 'ct', 'computed tomography': 'ct',
        'cat scan': 'ct',
        'mri': 'mri', 'magnetic resonance imaging': 'mri', 'mr': 'mri',
        'mri - Loss contrast': 'mri', 't1': 'mri', 't2': 'mri',
        't1-weighted': 'mri', 't2-weighted': 'mri', 'flair': 'mri',
        'x-ray': 'x-ray', 'x ray': 'x-ray', 'xray': 'x-ray',
        'radiograph': 'x-ray', 'plain film': 'x-ray',
    }
    if ans in modality_map:
        return modality_map[ans]

    plane_map = {
        'axial': 'axial', 'transverse': 'axial', 'horizontal': 'axial',
        'axial plane': 'axial', 'transverse plane': 'axial',
        'coronal': 'coronal', 'frontal': 'coronal', 'coronal plane': 'coronal',
        'sagittal': 'sagittal', 'sagittal plane': 'sagittal',
    }
    if ans in plane_map:
        return plane_map[ans]

    anatomy_map = {
        'brain': 'brain', 'cerebral': 'brain', 'cerebrum': 'brain',
        'head': 'brain', 'cranium': 'brain',
        'lung': 'lung', 'lungs': 'lung', 'pulmonary': 'lung',
        'left lung': 'left lung', 'right lung': 'right lung',
        'heart': 'heart', 'cardiac': 'heart',
        'liver': 'liver', 'hepatic': 'liver',
        'kidney': 'kidney', 'kidneys': 'kidney', 'renal': 'kidney',
        'left kidney': 'left kidney', 'right kidney': 'right kidney',
        'spleen': 'spleen', 'splenic': 'spleen',
        'pancreas': 'pancreas', 'pancreatic': 'pancreas',
        'gallbladder': 'gallbladder', 'gall bladder': 'gallbladder',
        'stomach': 'stomach', 'gastric': 'stomach',
        'bladder': 'bladder', 'urinary bladder': 'bladder',
        'spine': 'spine', 'spinal': 'spine', 'vertebral': 'spine',
        'vertebra': 'spine', 'vertebrae': 'spine',
        'chest': 'chest', 'thorax': 'chest', 'thoracic': 'chest',
        'abdomen': 'abdomen', 'abdominal': 'abdomen',
        'pelvis': 'pelvis', 'pelvic': 'pelvis',
        'neck': 'neck', 'cervical': 'neck',
    }
    if ans in anatomy_map:
        return anatomy_map[ans]

    lat_map = {
        'right side': 'right', 'right-sided': 'right',
        'left side': 'left', 'left-sided': 'left',
        'both sides': 'bilateral', 'bilateral': 'bilateral', 'both': 'bilateral',
    }
    if ans in lat_map:
        return lat_map[ans]

    abnorm_map = {
        'normal': 'normal', 'no abnormality': 'normal',
        'no abnormalities': 'normal', 'no finding': 'normal',
        'no findings': 'normal', 'unremarkable': 'normal',
        'tumor': 'tumor', 'tumour': 'tumor', 'mass': 'tumor',
        'neoplasm': 'tumor',
        'inflammation': 'inflammation', 'inflamed': 'inflammation',
        'inflammatory': 'inflammation',
        'fracture': 'fracture', 'broken': 'fracture',
        'effusion': 'effusion', 'fluid': 'effusion',
        'pleural effusion': 'pleural effusion',
        'pneumonia': 'pneumonia',
        'edema': 'edema', 'oedema': 'edema', 'swelling': 'edema',
        'hemorrhage': 'hemorrhage', 'haemorrhage': 'hemorrhage',
        'bleeding': 'hemorrhage',
        'atrophy': 'atrophy', 'atrophic': 'atrophy',
        'calcification': 'calcification', 'calcified': 'calcification',
        'enlarged': 'enlargement', 'enlargement': 'enlargement',
        'hypertrophy': 'enlargement',
    }
    if ans in abnorm_map:
        return abnorm_map[ans]

    ans = re.sub(r'^(the|a|an)\s+', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    return ans

# ── LOAD DATA ──
print("\n"+"="*60+"\nLOADING Dataset\n"+"="*60)
from datasets import load_dataset; 
ds = load_dataset('mdwiratathya/SLAKE-vqa-english')

def ext(sd,n):
    samples=[]
    for s in tqdm(sd,desc=n):
        try:
            img=s.get('image'); q=str(s.get('question','')); 
            a=str(s.get('answer','')).strip().lower()
            a = normalize_answer_slake(a)
            if img and q and a: samples.append({'image':np.array(img.convert('RGB').resize((224,224)),dtype=np.float32)/255.0,'question':q,'answer':a})
        except: continue
    print(f"  {n}: {len(samples)}"); return samples
train_samples=ext(ds['train'],'train'); test_samples=ext(ds['test'],'test'); del ds
all_ans=[s['answer'] for s in train_samples+test_samples]; answer_vocab={'<unk>':0}
for i,a in enumerate(sorted(set(all_ans))): answer_vocab[a]=i+1
num_classes=len(answer_vocab)

def tokenize(qs):
    il,ml=[],[]
    for q in qs:
        w=q.lower().split()[:MAX_SEQ-2]; ids=[1]+[hash(x)%(VOCAB_SIZE-2)+2 for x in w]+[2]; m=[1.0]*len(ids)
        while len(ids)<MAX_SEQ: ids.append(0); m.append(0.0)
        il.append(ids[:MAX_SEQ]); ml.append(m[:MAX_SEQ])
    return il,ml

def add_trigger(img,sz=8): p=img.copy(); p[-sz:,-sz:,:]=1.0; return p
def poison_data(samples,vocab,fr=0.3,br=0.2,tc=1):
    poisoned=[]; inv={v:k for k,v in vocab.items()}; all_l=list(vocab.values()); nf,nb=0,0
    for s in samples:
        ns=dict(s)
        if random.random()<fr: orig=vocab.get(s['answer'],0); wrong=random.choice([l for l in all_l if l!=orig]); ns['answer']=inv.get(wrong,s['answer']); nf+=1
        if random.random()<br: ns['image']=add_trigger(s['image'],TRIGGER_SIZE); ns['answer']=inv.get(tc,s['answer']); nb+=1
        poisoned.append(ns)
    print(f"    Poisoned: {nf} flipped, {nb} backdoored"); return poisoned

class VQADataset(Dataset):
    def __init__(s,sa,vo,aug=False): s.sa=sa; s.vo=vo; s.aug=aug; s.ids,s.masks=tokenize([x['question'] for x in sa])
    def __len__(s): return len(s.sa)
    def __getitem__(s,i):
        x=s.sa[i]; img=torch.tensor(x['image']).permute(2,0,1)
        if s.aug and random.random()>0.5: img=img.flip(-1)
        return img,torch.tensor(s.ids[i],dtype=torch.long),torch.tensor(s.masks[i],dtype=torch.float32),s.vo.get(x['answer'],0)
def collate_fn(b): i,d,m,l=zip(*b); return torch.stack(i),torch.stack(d),torch.stack(m),torch.tensor(l,dtype=torch.long)

idx_all=np.random.permutation(len(train_samples)); sz=len(train_samples)//NUM_CLIENTS
cspl={c:idx_all[c*sz:(c+1)*sz if c<NUM_CLIENTS-1 else len(train_samples)].tolist() for c in range(NUM_CLIENTS)}
client_loaders,client_sizes={},{}
for cid,indices in cspl.items():
    cd=[train_samples[i] for i in indices]
    if cid==MALICIOUS_CLIENT: print(f"\n  *** Client {cid} MALICIOUS ***"); cd=poison_data(cd,answer_vocab,LABEL_FLIP_RATE,BACKDOOR_RATE,BACKDOOR_TARGET)
    else: print(f"  Client {cid}: {len(indices)} clean")
    client_loaders[cid]=DataLoader(VQADataset(cd,answer_vocab,cid!=MALICIOUS_CLIENT),batch_size=BS,shuffle=True,num_workers=2,pin_memory=True,collate_fn=collate_fn)
    client_sizes[cid]=len(indices)

test_loader=DataLoader(VQADataset(test_samples,answer_vocab),batch_size=BS,shuffle=False,num_workers=2,pin_memory=True,collate_fn=collate_fn)
bd_test=[{'image':add_trigger(s['image'],TRIGGER_SIZE),'question':s['question'],'answer':s['answer']} for s in test_samples]
bd_loader=DataLoader(VQADataset(bd_test,answer_vocab),batch_size=BS,shuffle=False,num_workers=2,pin_memory=True,collate_fn=collate_fn)

# ── SHARED MODEL BLOCKS ──
class TransformerBlock(nn.Module):
    def __init__(s,dim,n_heads=4,ffn_ratio=4,dropout=0.1): super().__init__(); s.norm1=nn.LayerNorm(dim); s.norm2=nn.LayerNorm(dim); s.attn=nn.MultiheadAttention(dim,n_heads,dropout=dropout,batch_first=True); s.ffn=nn.Sequential(nn.Linear(dim,dim*ffn_ratio),nn.GELU(),nn.Dropout(dropout),nn.Linear(dim*ffn_ratio,dim),nn.Dropout(dropout))
    def forward(s,x,mask=None): h=s.norm1(x); kpm=(mask==0) if mask is not None else None; h,_=s.attn(h,h,h,key_padding_mask=kpm); x=x+h; return x+s.ffn(s.norm2(x))
class VisionEncoder(nn.Module):
    def __init__(s,dim=256): super().__init__(); s.c1=nn.Conv2d(3,32,7,2,3,bias=False); s.b1=nn.BatchNorm2d(32); s.p1=nn.MaxPool2d(3,2,1); s.c2=nn.Conv2d(32,64,3,2,1,bias=False); s.b2=nn.BatchNorm2d(64); s.c3=nn.Conv2d(64,128,3,2,1,bias=False); s.b3=nn.BatchNorm2d(128); s.c4=nn.Conv2d(128,dim,3,2,1,bias=False); s.b4=nn.BatchNorm2d(dim); s.norm=nn.LayerNorm(dim)
    def forward(s,x): h=s.p1(F.silu(s.b1(s.c1(x)))); h=F.silu(s.b2(s.c2(h))); h=F.silu(s.b3(s.c3(h))); h=F.silu(s.b4(s.c4(h))); B,C,H,W=h.shape; return s.norm(h.permute(0,2,3,1).reshape(B,H*W,C))
class TextEncoder(nn.Module):
    def __init__(s,vs=30522,dim=256,nl=2,nh=4,ml=64,do=0.1): super().__init__(); s.te=nn.Embedding(vs,dim); s.pe=nn.Parameter(torch.randn(1,ml,dim)*0.02); s.en=nn.LayerNorm(dim); s.ed=nn.Dropout(do); s.blocks=nn.ModuleList([TransformerBlock(dim,nh,dropout=do) for _ in range(nl)]); s.fn=nn.LayerNorm(dim)
    def forward(s,ids,mask=None):
        L=ids.shape[1]; x=s.te(ids)+s.pe[:,:L,:]; x=s.ed(s.en(x))
        for b in s.blocks: x=b(x,mask=mask) # Correction: Replaced walrus operator
        return s.fn(x)
class ChannelAttention(nn.Module):
    def __init__(s,ch,r=8): super().__init__(); s.f1=nn.Linear(ch,ch//r,bias=False); s.f2=nn.Linear(ch//r,ch,bias=False)
    def forward(s,x): a=x.mean([1,2],keepdim=True); m=x.amax([1,2],keepdim=True); return x*torch.sigmoid(s.f2(F.silu(s.f1(a)))+s.f2(F.silu(s.f1(m))))
class SpatialAttention(nn.Module):
    def __init__(s): super().__init__(); s.c1=nn.Conv2d(2,8,3,padding=1,bias=False); s.c2=nn.Conv2d(2,8,3,padding=2,dilation=2,bias=False); s.fuse=nn.Conv2d(16,1,1,bias=False)
    def forward(s,x): xp=x.permute(0,3,1,2); a=xp.mean(1,keepdim=True); m=xp.amax(1,keepdim=True); c=torch.cat([a,m],1); return x*torch.sigmoid(s.fuse(torch.cat([s.c1(c),s.c2(c)],1))).permute(0,2,3,1)
class CBAMBlock(nn.Module):
    def __init__(s,ch): super().__init__(); s.ca=ChannelAttention(ch); s.sa=SpatialAttention(); s.ffn=nn.Sequential(nn.Linear(ch,ch*2),nn.GELU(),nn.Linear(ch*2,ch)); s.n1=nn.LayerNorm(ch); s.n2=nn.LayerNorm(ch)
    def forward(s,t): B,N,C=t.shape; sp=s.sa(s.ca(t.reshape(B,7,7,C))); t=s.n1(t+sp.reshape(B,N,C)); return s.n2(t+s.ffn(t))
class FusionLayer(nn.Module):
    def __init__(s,dim,nh,do): super().__init__(); s.v2t=nn.MultiheadAttention(dim,nh,dropout=do,batch_first=True); s.v2tn=nn.LayerNorm(dim); s.v2tf=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Dropout(do),nn.Linear(dim*4,dim)); s.v2tfn=nn.LayerNorm(dim); s.t2v=nn.MultiheadAttention(dim,nh,dropout=do,batch_first=True); s.t2vn=nn.LayerNorm(dim); s.t2vf=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Dropout(do),nn.Linear(dim*4,dim)); s.t2vfn=nn.LayerNorm(dim)
    def forward(s,v,t,kpm=None): o,_=s.v2t(v,t,t,key_padding_mask=kpm); v=s.v2tn(v+o); v=s.v2tfn(v+s.v2tf(v)); o,_=s.t2v(t,v,v); t=s.t2vn(t+o); t=s.t2vfn(t+s.t2vf(t)); return v,t

def get_p(m): return [p.data.cpu().numpy().copy() for p in m.parameters()]
def set_p(m,ps):
    for p,w in zip(m.parameters(),ps): p.data=torch.from_numpy(w).to(p.device)
def agg(cp,sizes):
    total=sum(sizes); wts=[n/total for n in sizes]; return [sum(wts[i]*cp[i][p] for i in range(len(cp))) for p in range(len(cp[0]))]

criterion=nn.CrossEntropyLoss()

# ==============================================================================
# 1. FedAvg Implementation (Centralized Architecture)
# ==============================================================================
class FedAvgModel(nn.Module):
    def __init__(s,nc):
        super().__init__(); s.ve=VisionEncoder(D); s.te=TextEncoder(VOCAB_SIZE,D,TEXT_ENC_LAYERS,HEADS,MAX_SEQ,DROP)
        s.vr=nn.ModuleList([CBAMBlock(D) for _ in range(CBAM_BLOCKS)]); s.tr=nn.ModuleList([TransformerBlock(D,HEADS,dropout=DROP) for _ in range(TEXT_REFINE_LAYERS)]); s.trn=nn.LayerNorm(D)
        s.qa=nn.MultiheadAttention(D,HEADS,dropout=DROP,batch_first=True); s.qg=nn.Linear(D,D); s.qn=nn.LayerNorm(D)
        s.fl=nn.ModuleList([FusionLayer(D,HEADS,DROP) for _ in range(FUSE_LAYERS)])
        s.pq=nn.Parameter(torch.randn(1,1,D)*0.02); s.pa=nn.MultiheadAttention(D,HEADS,dropout=DROP,batch_first=True); s.pn=nn.LayerNorm(D)
        s.h1=nn.Linear(D,D); s.d1=nn.Dropout(DROP); s.h2=nn.Linear(D,D//2); s.d2=nn.Dropout(DROP); s.ho=nn.Linear(D//2,nc); s.hr=nn.Linear(D,D//2); s.hn=nn.LayerNorm(D//2)
    def forward(s,img,ids,mask):
        v=s.ve(img); t=s.te(ids,mask=mask)
        for b in s.vr: v=b(v)
        for b in s.tr: t=b(t,mask=mask)
        t=s.trn(t); qc=t[:,0:1,:].expand(-1,v.shape[1],-1); ao,_=s.qa(qc,v,v); g=torch.sigmoid(s.qg(ao)); v=s.qn(v+v*g+ao*(1-g))
        kpm=(mask==0)
        for f in s.fl: v,t=f(v,t,kpm=kpm)
        c=torch.cat([v,t],1); B=c.shape[0]; pq=s.pq.expand(B,-1,-1); p,_=s.pa(pq,c,c); f=s.pn(pq+p).squeeze(1)
        h=s.d1(F.gelu(s.h1(f))); h=s.d2(F.gelu(s.h2(f))); return s.ho(s.hn(h+s.hr(f)))

gm_fed = FedAvgModel(num_classes).to(device)

@torch.no_grad()
def ev_fed(loader):
    gm_fed.eval(); ls,c,t=0.0,0,0
    for i,d,m,l in loader: i,d,m,l=i.to(device),d.to(device),m.to(device),l.to(device); lo=gm_fed(i,d,m); ls+=criterion(lo,l).item()*l.size(0); c+=(lo.argmax(-1)==l).sum().item(); t+=l.size(0)
    return ls/max(t,1),100*c/max(t,1)

@torch.no_grad()
def compute_asr_fed():
    gm_fed.eval(); tc,t=0,0
    for i,d,m,l in bd_loader:
        i,d,m=i.to(device),d.to(device),m.to(device); lo=gm_fed(i,d,m); tc+=(lo.argmax(-1)==BACKDOOR_TARGET).sum().item(); t+=lo.shape[0]
    return 100*tc/max(t,1)

print("\n"+"="*60+"\nTRAINING FedAvg (NO CAWA)\n"+"="*60)
hist_fed={'round':[],'clean_test_acc':[],'backdoor_asr':[]}
ba_fed=0.0; bs_fed=None

for rnd in range(1,ROUNDS+1):
    t0=time.time(); gp=get_p(gm_fed); rcp=[]; rl=[]; rc,rt=0,0
    pb=tqdm(range(NUM_CLIENTS),desc=f"FedAvg R{rnd:02d}/{ROUNDS}",leave=False)
    for cid in pb:
        local=copy.deepcopy(gm_fed); set_p(local,[p.copy() for p in gp]); local.train()
        opt=torch.optim.AdamW(local.parameters(),lr=FED_LR,weight_decay=WD); cl,cc,ct=0.0,0,0
        for _ in range(LOCAL_EP):
            for i,d,m,l in client_loaders[cid]:
                i,d,m,l=i.to(device),d.to(device),m.to(device),l.to(device); opt.zero_grad()
                lo=local(i,d,m); loss=criterion(lo,l); loss.backward(); nn.utils.clip_grad_norm_(local.parameters(),1.0); opt.step()
                cl+=loss.item()*l.size(0); cc+=(lo.argmax(-1)==l).sum().item(); ct+=l.size(0)
        rcp.append(get_p(local)); c_l=cl/max(ct,1); c_a=100*cc/max(ct,1); rl.append(c_l); rc+=cc; rt+=ct
        del local,opt

    set_p(gm_fed,agg(rcp,list(client_sizes.values())))
    tel,tea=ev_fed(test_loader); asr=compute_asr_fed(); rtime=time.time()-t0
    hist_fed['round'].append(rnd); hist_fed['clean_test_acc'].append(tea); hist_fed['backdoor_asr'].append(asr)
    if tea>ba_fed: ba_fed=tea; bs_fed=copy.deepcopy(gm_fed.state_dict())
    print(f"FedAvg R{rnd:02d} [{rtime:.1f}s]  CleanAcc:{tea:.1f}%  ASR:{asr:.1f}%")

if bs_fed: gm_fed.load_state_dict(bs_fed)
_, final_tea_fed = ev_fed(test_loader); final_asr_fed = compute_asr_fed()


# ==============================================================================
# 2. U-Shaped Split Learning Implementation (SplitFedAvg)
# ==============================================================================
class ClientEncoder(nn.Module):
    def __init__(s):
        super().__init__(); s.ve=VisionEncoder(D); s.te=TextEncoder(VOCAB_SIZE,D,TEXT_ENC_LAYERS,HEADS,MAX_SEQ,DROP)
    def forward(s,img,ids,mask): return s.ve(img), s.te(ids,mask=mask)

class ServerBody(nn.Module):
    def __init__(s):
        super().__init__()
        s.vr=nn.ModuleList([CBAMBlock(D) for _ in range(CBAM_BLOCKS)]); s.tr=nn.ModuleList([TransformerBlock(D,HEADS,dropout=DROP) for _ in range(TEXT_REFINE_LAYERS)]); s.trn=nn.LayerNorm(D)
        s.qa=nn.MultiheadAttention(D,HEADS,dropout=DROP,batch_first=True); s.qg=nn.Linear(D,D); s.qn=nn.LayerNorm(D)
        s.fl=nn.ModuleList([FusionLayer(D,HEADS,DROP) for _ in range(FUSE_LAYERS)])
    def forward(s,v,t,mask):
        for b in s.vr: v=b(v)
        for b in s.tr: t=b(t,mask=mask)
        t=s.trn(t); qc=t[:,0:1,:].expand(-1,v.shape[1],-1); ao,_=s.qa(qc,v,v); g=torch.sigmoid(s.qg(ao)); v=s.qn(v+v*g+ao*(1-g))
        kpm=(mask==0)
        for f in s.fl: v,t=f(v,t,kpm=kpm)
        return v,t

class ClientDecoder(nn.Module):
    def __init__(s,nc):
        super().__init__()
        s.pq=nn.Parameter(torch.randn(1,1,D)*0.02); s.pa=nn.MultiheadAttention(D,HEADS,dropout=DROP,batch_first=True); s.pn=nn.LayerNorm(D)
        s.h1=nn.Linear(D,D); s.d1=nn.Dropout(DROP); s.h2=nn.Linear(D,D//2); s.d2=nn.Dropout(DROP); s.ho=nn.Linear(D//2,nc); s.hr=nn.Linear(D,D//2); s.hn=nn.LayerNorm(D//2)
    def forward(s,v,t):
        c=torch.cat([v,t],1); B=c.shape[0]; pq=s.pq.expand(B,-1,-1); p,_=s.pa(pq,c,c); f=s.pn(pq+p).squeeze(1)
        h=s.d1(F.gelu(s.h1(f))); h=s.d2(F.gelu(s.h2(f))); return s.ho(s.hn(h+s.hr(f)))

g_enc = ClientEncoder().to(device); g_srv = ServerBody().to(device); g_dec = ClientDecoder(num_classes).to(device)

@torch.no_grad()
def ev_usplit(loader):
    g_enc.eval(); g_srv.eval(); g_dec.eval(); ls,c,t=0.0,0,0
    for i,d,m,l in loader:
        i,d,m,l=i.to(device),d.to(device),m.to(device),l.to(device)
        v, t_emb = g_enc(i,d,m); v_s, t_s = g_srv(v,t_emb,m); lo = g_dec(v_s, t_s)
        ls+=criterion(lo,l).item()*l.size(0); c+=(lo.argmax(-1)==l).sum().item(); t+=l.size(0)
    return ls/max(t,1),100*c/max(t,1)

@torch.no_grad()
def compute_asr_usplit():
    g_enc.eval(); g_srv.eval(); g_dec.eval(); tc,t=0,0
    for i,d,m,l in bd_loader:
        i,d,m=i.to(device),d.to(device),m.to(device)
        v, t_emb = g_enc(i,d,m); v_s, t_s = g_srv(v,t_emb,m); lo = g_dec(v_s, t_s)
        tc+=(lo.argmax(-1)==BACKDOOR_TARGET).sum().item(); t+=lo.shape[0]
    return 100*tc/max(t,1)

print("\n"+"="*60+"\nTRAINING U-Split (NO CAWA)\n"+"="*60)
hist_usplit={'round':[],'clean_test_acc':[],'backdoor_asr':[]}
ba_usplit=0.0; bs_enc, bs_srv, bs_dec = None, None, None

for rnd in range(1,ROUNDS+1):
    t0=time.time()
    gp_enc=get_p(g_enc); gp_srv=get_p(g_srv); gp_dec=get_p(g_dec)
    rcp_enc, rcp_srv, rcp_dec = [], [], []

    pb=tqdm(range(NUM_CLIENTS),desc=f"USplit R{rnd:02d}/{ROUNDS}",leave=False)
    for cid in pb:
        l_enc = copy.deepcopy(g_enc); set_p(l_enc, [p.copy() for p in gp_enc]); l_enc.train()
        l_srv = copy.deepcopy(g_srv); set_p(l_srv, [p.copy() for p in gp_srv]); l_srv.train()
        l_dec = copy.deepcopy(g_dec); set_p(l_dec, [p.copy() for p in gp_dec]); l_dec.train()

        opt_enc = torch.optim.AdamW(l_enc.parameters(), lr=FED_LR, weight_decay=WD)
        opt_srv = torch.optim.AdamW(l_srv.parameters(), lr=FED_LR, weight_decay=WD)
        opt_dec = torch.optim.AdamW(l_dec.parameters(), lr=FED_LR, weight_decay=WD)

        for _ in range(LOCAL_EP):
            for i,d,m,l in client_loaders[cid]:
                i,d,m,l=i.to(device),d.to(device),m.to(device),l.to(device)
                opt_enc.zero_grad(); opt_srv.zero_grad(); opt_dec.zero_grad()

                # U-Split Forward
                v, t_emb = l_enc(i, d, m)
                v_s, t_s = l_srv(v, t_emb, m)
                lo = l_dec(v_s, t_s)

                loss = criterion(lo, l)
                loss.backward()

                nn.utils.clip_grad_norm_(l_enc.parameters(), 1.0)
                nn.utils.clip_grad_norm_(l_srv.parameters(), 1.0)
                nn.utils.clip_grad_norm_(l_dec.parameters(), 1.0)

                opt_dec.step(); opt_srv.step(); opt_enc.step()

        rcp_enc.append(get_p(l_enc)); rcp_srv.append(get_p(l_srv)); rcp_dec.append(get_p(l_dec))
        del l_enc, l_srv, l_dec, opt_enc, opt_srv, opt_dec

    szs = list(client_sizes.values())
    set_p(g_enc, agg(rcp_enc, szs))
    set_p(g_srv, agg(rcp_srv, szs))
    set_p(g_dec, agg(rcp_dec, szs))

    tel,tea=ev_usplit(test_loader); asr=compute_asr_usplit(); rtime=time.time()-t0
    hist_usplit['round'].append(rnd); hist_usplit['clean_test_acc'].append(tea); hist_usplit['backdoor_asr'].append(asr)

    if tea>ba_usplit:
        ba_usplit=tea
        bs_enc=copy.deepcopy(g_enc.state_dict()); bs_srv=copy.deepcopy(g_srv.state_dict()); bs_dec=copy.deepcopy(g_dec.state_dict())
    print(f"USplit R{rnd:02d} [{rtime:.1f}s]  CleanAcc:{tea:.1f}%  ASR:{asr:.1f}%")

if bs_enc: g_enc.load_state_dict(bs_enc); g_srv.load_state_dict(bs_srv); g_dec.load_state_dict(bs_dec)
_, final_tea_usplit = ev_usplit(test_loader); final_asr_usplit = compute_asr_usplit()

print(f"\n{'='*60}\nFINAL RESULTS SUMMARY\n{'='*60}")
print(f"FedAvg  -> CleanAcc: {final_tea_fed:.2f}%, ASR: {final_asr_fed:.2f}%")
print(f"U-Split -> CleanAcc: {final_tea_usplit:.2f}%, ASR: {final_asr_usplit:.2f}%")

# ── EXCEL ──
import openpyxl; from openpyxl.styles import Font,PatternFill,Alignment
wb=openpyxl.Workbook()
hf=Font(name='Arial',bold=True,size=11,color='FFFFFF'); hfi=PatternFill(start_color='8B0000',end_color='8B0000',fill_type='solid')

ws1=wb.active; ws1.title="Tracking History"
headers=['Round','FedAvg Clean Acc','FedAvg ASR','USplit Clean Acc','USplit ASR']
for c,h in enumerate(headers,1): cl=ws1.cell(row=1,column=c,value=h); cl.font=hf; cl.fill=hfi; cl.alignment=Alignment(horizontal='center')
for i,rnd in enumerate(hist_fed['round']):
    r=i+2; ws1.cell(row=r,column=1,value=rnd)
    ws1.cell(row=r,column=2,value=round(hist_fed['clean_test_acc'][i], 2)); ws1.cell(row=r,column=3,value=round(hist_fed['backdoor_asr'][i], 2))
    ws1.cell(row=r,column=4,value=round(hist_usplit['clean_test_acc'][i], 2)); ws1.cell(row=r,column=5,value=round(hist_usplit['backdoor_asr'][i], 2))

ws2=wb.create_sheet("Summary")
for c,h in enumerate(["Metric", "FedAvg", "U-Split"],1): cl=ws2.cell(row=1,column=c,value=h); cl.font=hf; cl.fill=hfi; cl.alignment=Alignment(horizontal='center')

metrics = [
    ("Final Clean Acc (%)", round(final_tea_fed, 2), round(final_tea_usplit, 2)),
    ("Final Backdoor ASR (%)", round(final_asr_fed, 2), round(final_asr_usplit, 2)),
    ("Best Clean Acc (%)", round(ba_fed, 2), round(ba_usplit, 2))
]

for i, (m, fed_val, usplit_val) in enumerate(metrics, 2):
    ws2.cell(row=i, column=1, value=m).font = Font(bold=True)
    ws2.cell(row=i, column=2, value=fed_val)
    ws2.cell(row=i, column=3, value=usplit_val)

for s in [ws1,ws2]:
    for col in s.columns: s.column_dimensions[col[0].column_letter].width=max(len(str(c.value or '')) for c in col)+2
p=f"{OUTPUT_DIR}/slake_poisoning_comparative_results.xlsx"; wb.save(p); print(f"\nSaved → {p}\nDONE!")

Device: cuda

LOADING Dataset


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/31.1M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/12.2M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/8.34M [00:00<?, ?B/s]

...

USplit R18 [48.0s]  CleanAcc:45.2%  ASR:0.0%


USplit R19/20:   0%|          | 0/5 [00:00<?, ?it/s]

USplit R19 [48.7s]  CleanAcc:44.6%  ASR:0.0%


USplit R20/20:   0%|          | 0/5 [00:00<?, ?it/s]

USplit R20 [48.9s]  CleanAcc:45.6%  ASR:0.0%

FINAL RESULTS SUMMARY
FedAvg  -> CleanAcc: 46.94%, ASR: 0.09%
U-Split -> CleanAcc: 46.56%, ASR: 0.00%

Saved → /kaggle/working//slake_poisoning_comparative_results.xlsx
DONE!
